# 6.2 — Data Quality Review

Run this notebook in order. Earlier sections are included as prerequisites so this notebook can run in a fresh Colab session. Upload the requested source files when prompted. Source: https://github.com/paperwave/RAGent

## 1. Data acquisition and source identification

In [ ]:
from pathlib import Path
import pandas as pd
import json, zipfile, shutil
from IPython.display import display
WORK = Path.cwd() / 'ai687_project'
WORK.mkdir(exist_ok=True)
def require_file(name):
    p = WORK / name
    if not p.exists():
        if (Path.cwd()/name).exists():
            shutil.copy2(Path.cwd()/name,p)
        else:
            from google.colab import files
            print('Upload:', name)
            uploaded = files.upload()
            if name not in uploaded:
                raise ValueError('Required file: '+name)
            p.write_bytes(uploaded[name])
    return p
with zipfile.ZipFile(require_file('RAGent-main.zip')) as z:
    raw = pd.read_csv(z.open('RAGent-main/data/document_folds/acre.csv'))
    acp = pd.read_csv(z.open('RAGent-main/data/document_folds/acre_acp.csv'))
print('Source records:',len(raw),'Access-control records:',len(acp))
print(acp.dtypes)
display(acp.head())
assert len(raw)==583 and len(acp)==433


## 2. Data quality review

In [ ]:
text=acp['input'].astype('string')
quality={'records':len(acp),'missing':int(text.isna().sum()),'blank':int(text.str.strip().eq('').sum()),'duplicates':int(text.dropna().str.strip().duplicated().sum())}
print(quality)
assert not any(quality[k] for k in ['missing','blank','duplicates'])
full=pd.read_csv(require_file('acre_prepared.csv'))
assert full.policy_id.is_unique and len(full)==len(acp)
assert full.policy_text.str.strip().tolist()==text.str.strip().tolist()
